# Core Nexus: Graph DB 101
ยินดีต้อนรับสู่ Workshop Neo4j ครับ! ในไฟล์นี้เราจะมาทดลองเขียน Python เพื่อ:
1. เชื่อมต่อไปยังฐานข้อมูล Neo4j
2. นำเข้าข้อมูล (Ingest)
3. ดึงข้อมูล (Query)

In [ ]:
from neo4j import GraphDatabase

URI = "bolt://neo4j:7687"
AUTH = ("neo4j", "nexus2026")

driver = GraphDatabase.driver(URI, auth=AUTH)
driver.verify_connectivity()
print("เชื่อมต่อ Neo4j สำเร็จ พร้อมลุย!")

In [ ]:
def setup_infrastructure_graph(tx):
    query = """
    CREATE (bkk:City {name: 'Bangkok'})
    CREATE (cnx:City {name: 'Chiang Mai'})
    CREATE (pkt:City {name: 'Phuket'})
    CREATE (tyo:City {name: 'Tokyo'})
    CREATE (nyc:City {name: 'New York'})

    CREATE (alice:Person {name: 'Alice', age: 28})
    CREATE (bob:Person {name: 'Bob', age: 32})
    CREATE (charlie:Person {name: 'Charlie', age: 25})
    CREATE (david:Person {name: 'David', age: 35})
    CREATE (eve:Person {name: 'Eve', age: 29})
    CREATE (frank:Person {name: 'Frank', age: 40})
    CREATE (grace:Person {name: 'Grace', age: 22})
    CREATE (heidi:Person {name: 'Heidi', age: 27})
    CREATE (ivan:Person {name: 'Ivan', age: 31})
    CREATE (judy:Person {name: 'Judy', age: 26})

    CREATE (matrix:Movie {title: 'The Matrix', year: 1999})
    CREATE (inception:Movie {title: 'Inception', year: 2010})
    CREATE (avatar:Movie {title: 'Avatar', year: 2009})
    CREATE (interstellar:Movie {title: 'Interstellar', year: 2014})
    CREATE (parasite:Movie {title: 'Parasite', year: 2019})
    CREATE (avengers:Movie {title: 'The Avengers', year: 2012})

    CREATE (padthai:Restaurant {name: 'Pad Thai Go', rating: 4.5})
    CREATE (sushi:Restaurant {name: 'Sushi Zen', rating: 4.8})
    CREATE (somtum:Restaurant {name: 'Somtum Nua', rating: 4.6})
    CREATE (ramen:Restaurant {name: 'Ramen King', rating: 4.9})
    CREATE (burger:Restaurant {name: 'Burger Boss', rating: 4.2})
    CREATE (vegan:Restaurant {name: 'Green Bowl', rating: 4.7})

    CREATE (photo:Hobby {name: 'Photography'})
    CREATE (game:Hobby {name: 'Gaming'})
    CREATE (cook:Hobby {name: 'Cooking'})
    CREATE (travel:Hobby {name: 'Traveling'})

    CREATE (alice)-[:LIVES_IN]->(bkk)
    CREATE (bob)-[:LIVES_IN]->(bkk)
    CREATE (charlie)-[:LIVES_IN]->(cnx)
    CREATE (david)-[:LIVES_IN]->(pkt)
    CREATE (eve)-[:LIVES_IN]->(bkk)
    CREATE (frank)-[:LIVES_IN]->(tyo)
    CREATE (grace)-[:LIVES_IN]->(nyc)
    CREATE (heidi)-[:LIVES_IN]->(cnx)
    CREATE (ivan)-[:LIVES_IN]->(tyo)
    CREATE (judy)-[:LIVES_IN]->(bkk)

    CREATE (alice)-[:KNOWS]->(bob)
    CREATE (bob)-[:KNOWS]->(charlie)
    CREATE (alice)-[:KNOWS]->(eve)
    CREATE (charlie)-[:KNOWS]->(david)
    CREATE (eve)-[:KNOWS]->(david)
    CREATE (alice)-[:KNOWS]->(grace)
    CREATE (frank)-[:KNOWS]->(ivan)
    CREATE (heidi)-[:KNOWS]->(charlie)
    CREATE (judy)-[:KNOWS]->(alice)
    CREATE (judy)-[:KNOWS]->(bob)

    CREATE (alice)-[:LIKES]->(matrix)
    CREATE (bob)-[:LIKES]->(inception)
    CREATE (charlie)-[:LIKES]->(matrix)
    CREATE (eve)-[:LIKES]->(avatar)
    CREATE (frank)-[:LIKES]->(parasite)
    CREATE (grace)-[:LIKES]->(interstellar)
    CREATE (heidi)-[:LIKES]->(avengers)
    CREATE (ivan)-[:LIKES]->(inception)
    CREATE (judy)-[:LIKES]->(interstellar)

    CREATE (alice)-[:VISITED {times: 3}]->(padthai)
    CREATE (bob)-[:VISITED {times: 1}]->(sushi)
    CREATE (charlie)-[:VISITED {times: 5}]->(somtum)
    CREATE (david)-[:VISITED {times: 2}]->(burger)
    CREATE (eve)-[:VISITED {times: 4}]->(vegan)
    CREATE (frank)-[:VISITED {times: 10}]->(ramen)
    CREATE (grace)-[:VISITED {times: 2}]->(vegan)
    CREATE (judy)-[:VISITED {times: 6}]->(padthai)

    CREATE (padthai)-[:LOCATED_IN]->(bkk)
    CREATE (sushi)-[:LOCATED_IN]->(bkk)
    CREATE (somtum)-[:LOCATED_IN]->(cnx)
    CREATE (ramen)-[:LOCATED_IN]->(tyo)
    CREATE (burger)-[:LOCATED_IN]->(pkt)
    CREATE (vegan)-[:LOCATED_IN]->(nyc)

    CREATE (alice)-[:HAS_HOBBY]->(photo)
    CREATE (alice)-[:HAS_HOBBY]->(travel)
    CREATE (bob)-[:HAS_HOBBY]->(game)
    CREATE (charlie)-[:HAS_HOBBY]->(cook)
    CREATE (david)-[:HAS_HOBBY]->(travel)
    CREATE (eve)-[:HAS_HOBBY]->(cook)
    CREATE (frank)-[:HAS_HOBBY]->(photo)
    CREATE (grace)-[:HAS_HOBBY]->(travel)
    CREATE (ivan)-[:HAS_HOBBY]->(game)
    CREATE (judy)-[:HAS_HOBBY]->(photo)
    """
    tx.run(query)
    print("นำเข้าข้อมูล (Ingest) โครงสร้าง Social Network (30+ Nodes) เข้าสู่ Graph สำเร็จ!")

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    session.execute_write(setup_infrastructure_graph)

**Tip:** ลองสลับไปเปิดหน้า **Neo4j Browser** ที่พอร์ต `7474` แล้วรันคำสั่ง `MATCH path = (m)-[:LIKES]->(j:Movie {title: 'The Matrix'}) RETURN path` แล้วกดปุ่ม Graph เพื่อดูภาพกราฟสวยๆ ได้เลยนะฮะ!

In [ ]:
def query_judy_likes(tx):
    query = """
    MATCH (j:Person {name: 'Judy'})-[:LIKES]->(m:Movie)
    RETURN j.name AS person, m.title AS movie
    """
    result = tx.run(query)
    for record in result:
        print(f"พบว่า {record['person']} ชื่นชอบภาพยนตร์เรื่อง: {record['movie']}")

print("ผลลัพธ์การ Query:")
with driver.session() as session:
    session.execute_read(query_judy_likes)